# 01 — Traffic EDA

Exploratory Data Analysis for the PEMS-BAY dataset.

**Dataset:** PEMS-BAY Bay Area traffic speed readings, Jan–Jun 2017
**Sensors:** 325 loop detectors
**Timestamps:** 52,116 (5-minute intervals)
**Measurement:** Traffic speed in mph

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sys.path.insert(0, str(Path('.').resolve().parent))
from config.settings import CLEANED_DATA_PATH, FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use('dark_background')
sns.set_theme(style='darkgrid')
print('Libraries loaded.')

## 1. Load & Inspect Dataset

In [ ]:
# Load cleaned long-format data
df = pd.read_csv(CLEANED_DATA_PATH, parse_dates=['timestamp'])
df['sensor_id'] = df['sensor_id'].astype(str)
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['timestamp'].dt.month
df['date'] = df['timestamp'].dt.date

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nDate range: {df.timestamp.min()} → {df.timestamp.max()}')
print(f'Sensors: {df.sensor_id.nunique()}')
print(f'\nDtypes:\n{df.dtypes}')

In [ ]:
# Basic statistics
print('=== Speed Statistics ===')
print(df['observed_speed'].describe())
print(f'\nMissing values: {df.isnull().sum().to_dict()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate (timestamp, sensor_id): {df.duplicated(["timestamp","sensor_id"]).sum()}')

## 2. Speed Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['observed_speed'], bins=100, color='#00D4FF', edgecolor='none', alpha=0.8)
axes[0].set_title('Speed Distribution', fontsize=14)
axes[0].set_xlabel('Speed (mph)'); axes[0].set_ylabel('Frequency')
axes[0].axvline(df['observed_speed'].mean(), color='#FF8C00', linestyle='--', label=f'Mean: {df["observed_speed"].mean():.1f}')
axes[0].axvline(df['observed_speed'].median(), color='#00C851', linestyle='--', label=f'Median: {df["observed_speed"].median():.1f}')
axes[0].legend()

# Box plot
axes[1].boxplot(df['observed_speed'], patch_artist=True, boxprops=dict(facecolor='#00D4FF', alpha=0.6))
axes[1].set_title('Speed Box Plot', fontsize=14)
axes[1].set_ylabel('Speed (mph)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_speed_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {FIGURES_DIR / "01_speed_distribution.png"}')

## 3. Hourly & Daily Patterns

In [ ]:
hourly = df.groupby('hour')['observed_speed'].agg(['mean','std']).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(hourly['hour'], hourly['mean']-hourly['std'], hourly['mean']+hourly['std'],
                alpha=0.3, color='#00D4FF', label='Mean ± 1 Std')
ax.plot(hourly['hour'], hourly['mean'], color='#00D4FF', linewidth=2.5, label='Mean Speed')
ax.axvspan(7, 9, alpha=0.15, color='#FF1744', label='AM Peak (7–9)')
ax.axvspan(17, 19, alpha=0.15, color='#FF8C00', label='PM Peak (17–19)')
ax.set_title('Average Speed by Hour of Day (All Sensors)', fontsize=14)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Speed (mph)')
ax.set_xticks(range(0,24,2))
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_hourly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
dow_map = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
daily = df.groupby('day_of_week')['observed_speed'].mean().reset_index()
daily['day_name'] = daily['day_of_week'].map(dow_map)
colors = ['#00D4FF']*5 + ['#FFD700']*2

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(daily['day_name'], daily['observed_speed'], color=colors)
ax.set_title('Average Speed by Day of Week', fontsize=14)
ax.set_xlabel('Day'); ax.set_ylabel('Avg Speed (mph)')
ax.bar_label(bars, fmt='%.1f', padding=3)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#00D4FF',label='Weekday'), Patch(color='#FFD700',label='Weekend')])
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_daily_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Sensor-wise Statistics

In [ ]:
sensor_stats = df.groupby('sensor_id')['observed_speed'].agg(
    mean='mean', std='std', min='min', max='max', count='count'
).reset_index().sort_values('mean')

print('=== Sensor Statistics Summary ===')
print(sensor_stats.describe())

print('\n=== 10 Lowest Avg Speed Sensors ===')
print(sensor_stats.head(10).to_string(index=False))

print('\n=== 10 Highest Avg Speed Sensors ===')
print(sensor_stats.tail(10).to_string(index=False))

In [ ]:
# Sensor mean speed distribution
fig, ax = plt.subplots(figsize=(12,5))
ax.hist(sensor_stats['mean'], bins=40, color='#00D4FF', edgecolor='none', alpha=0.8)
ax.set_title('Distribution of Per-Sensor Average Speed', fontsize=14)
ax.set_xlabel('Average Speed (mph)'); ax.set_ylabel('Number of Sensors')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_sensor_mean_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Heatmap — Hour × Day of Week

In [ ]:
pivot = df.groupby(['day_of_week','hour'])['observed_speed'].mean().unstack()
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, ax = plt.subplots(figsize=(18, 5))
sns.heatmap(pivot, cmap='RdYlGn', linewidths=0, ax=ax,
            yticklabels=dow_labels, cbar_kws={'label':'Avg Speed (mph)'})
ax.set_title('Avg Speed Heatmap — Hour of Day × Day of Week', fontsize=14)
ax.set_xlabel('Hour of Day'); ax.set_ylabel('Day of Week')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_heatmap_hour_dow.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Outlier Analysis

In [ ]:
outlier_pct = df['outlier_flag'].mean() * 100
print(f'Overall outlier rate: {outlier_pct:.2f}%')
print(f'Total outlier readings: {df["outlier_flag"].sum():,}')
print()

# Z-score outliers
from scipy import stats
zscores = np.abs(stats.zscore(df['observed_speed']))
print(f'Z-score > 3 outliers: {(zscores > 3).sum():,}  ({(zscores>3).mean()*100:.2f}%)')

# IQR bounds
q1, q3 = df['observed_speed'].quantile([0.25, 0.75])
iqr = q3 - q1
lower = q1 - 3*iqr
upper = q3 + 3*iqr
print(f'IQR bounds: [{lower:.1f}, {upper:.1f}] mph')
print()
print('Handling decision: Outlier values are FLAGGED (outlier_flag=1) but NOT removed.')
print('Rationale: Extreme values may represent real congestion events or incidents.')
print('Models are trained without filtering; outlier_flag is available as a feature.')

## 7. Weekday vs Weekend Comparison

In [ ]:
# Box plots
wk = df[df['is_weekend']==0]['observed_speed'].sample(50000, random_state=42)
we = df[df['is_weekend']==1]['observed_speed'].sample(20000, random_state=42)

fig, ax = plt.subplots(figsize=(8,6))
bp = ax.boxplot([wk, we], labels=['Weekday','Weekend'], patch_artist=True,
                boxprops=dict(facecolor='#00D4FF', alpha=0.6),
                medianprops=dict(color='#FF8C00', linewidth=2))
bp['boxes'][1].set_facecolor('#00C851')
ax.set_title('Speed: Weekday vs Weekend', fontsize=14)
ax.set_ylabel('Speed (mph)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_weekday_weekend.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Weekday mean: {wk.mean():.2f} mph')
print(f'Weekend mean: {we.mean():.2f} mph')

## 8. Summary Statistics Export

In [ ]:
from pathlib import Path
REPORTS_DIR = Path('..') / 'reports'

summary = {
    'total_rows': len(df),
    'sensors': df['sensor_id'].nunique(),
    'date_start': str(df['timestamp'].min()),
    'date_end': str(df['timestamp'].max()),
    'avg_speed': round(float(df['observed_speed'].mean()), 3),
    'median_speed': round(float(df['observed_speed'].median()), 3),
    'std_speed': round(float(df['observed_speed'].std()), 3),
    'min_speed': round(float(df['observed_speed'].min()), 3),
    'max_speed': round(float(df['observed_speed'].max()), 3),
    'outlier_pct': round(float(df['outlier_flag'].mean()*100), 2),
    'missing_pct': 0.0,
}

with open(REPORTS_DIR / 'eda_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

sensor_stats.to_csv(REPORTS_DIR / 'sensor_stats.csv', index=False)

print('Saved EDA outputs:')
for k, v in summary.items():
    print(f'  {k:<25} {v}')